# Readability Dataset Demo
## Sentence-Level Readability Assessment Data

This notebook demonstrates how to load and explore the readability datasets for sentence-level readability assessment.

### What this artifact provides:
- **CLEAR Corpus**: 3,543 reading passage excerpts with multiple readability metrics
- **Agentlans Readability Dataset**: 2,000 sampled paragraphs with continuous grade level scores
- Both datasets are standardized to `exp_sel_data_out.json` schema with:
  - `input`: text (sentence/excerpt)
  - `output`: normalized readability score (0.0 = easiest, 1.0 = hardest)
  - Additional metadata fields

### Demo Overview:
1. Load the demo dataset from GitHub (with local fallback)
2. Explore the data structure
3. Visualize readability score distributions
4. Analyze text characteristics

In [ ]:
# Install dependencies (Colab-compatible pattern)
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print("Dependencies ready.")

In [ ]:
# Imports
import json
import os
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

print("Imports successful.")

In [ ]:
# Data loading helper with GitHub URL and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-fea63a-the-uniformity-principle-how-within-sent/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    # Try GitHub URL first
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print(f"Loaded data from GitHub URL")
            return data
    except Exception as e:
        print(f"GitHub URL failed: {e}")
    
    # Fall back to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print(f"Loaded data from local file")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or locally")

print("Data loading function defined.")

In [ ]:
# Load the demo data
data = load_data()

# Display basic info
print(f"Number of datasets: {len(data['datasets'])}")
for i, dataset in enumerate(data['datasets']):
    print(f"\nDataset {i+1}: {dataset['dataset']}")
    print(f"  Number of examples: {len(dataset['examples'])}")

## Data Exploration

Let's examine the structure of our readability dataset and understand the fields available.

In [ ]:
# Explore the data structure
dataset = data['datasets'][0]  # Get first dataset
examples = dataset['examples']

print(f"Dataset name: {dataset['dataset']}")
print(f"Number of examples: {len(examples)}")
print("\n" + "="*60)
print("First example structure:")
print("="*60)
for key, value in examples[0].items():
    if key == 'input':
        print(f"{key}: {str(value)[:100]}...")
    else:
        print(f"{key}: {value}")

## Visualize Readability Scores

The `output` field contains normalized readability scores (0.0 = easiest, 1.0 = hardest).
Let's visualize the distribution of scores in our demo dataset.

In [ ]:
# Extract and visualize readability scores
scores = [float(ex['output']) for ex in examples]
texts = [ex['input'] for ex in examples]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of scores
axes[0].bar(range(len(scores)), scores, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Example Index')
axes[0].set_ylabel('Readability Score (0=easy, 1=hard)')
axes[0].set_title('Readability Scores for Demo Examples')
axes[0].grid(True, alpha=0.3)

# Histogram of scores (for larger datasets)
axes[1].hist(scores, bins=10, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Readability Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Readability Scores')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print score statistics
print(f"\nScore Statistics:")
print(f"  Min score: {min(scores):.3f}")
print(f"  Max score: {max(scores):.3f}")
print(f"  Mean score: {np.mean(scores):.3f}")
print(f"  Std deviation: {np.std(scores):.3f}")

## Text Characteristics Analysis

Let's analyze the text characteristics and see how they relate to readability scores.
We'll look at:
- Text length (word count)
- Sentence length
- Relationship with readability scores

In [ ]:
# Analyze text characteristics
import re

def count_words(text):
    """Count words in text."""
    words = re.findall(r'\b\w+\b', text)
    return len(words)

def count_sentences(text):
    """Count sentences in text."""
    sentences = re.split(r'[.!?]+', text)
    return len([s for s in sentences if s.strip()])

# Calculate statistics
word_counts = [count_words(ex['input']) for ex in examples]
sentence_counts = [count_sentences(ex['input']) for ex in examples]

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Word count vs score
axes[0, 0].scatter(word_counts, scores, color='steelblue', alpha=0.6, s=100)
axes[0, 0].set_xlabel('Word Count')
axes[0, 0].set_ylabel('Readability Score')
axes[0, 0].set_title('Word Count vs Readability Score')
axes[0, 0].grid(True, alpha=0.3)

# Sentence count vs score
axes[0, 1].scatter(sentence_counts, scores, color='coral', alpha=0.6, s=100)
axes[0, 1].set_xlabel('Sentence Count')
axes[0, 1].set_ylabel('Readability Score')
axes[0, 1].set_title('Sentence Count vs Readability Score')
axes[0, 1].grid(True, alpha=0.3)

# Word count distribution
axes[1, 0].bar(range(len(word_counts)), word_counts, color='steelblue', alpha=0.7)
axes[1, 0].set_xlabel('Example Index')
axes[1, 0].set_ylabel('Word Count')
axes[1, 0].set_title('Word Count per Example')
axes[1, 0].grid(True, alpha=0.3)

# Score vs avg word length (approximate)
avg_word_lengths = [np.mean([len(w) for w in ex['input'].split()]) if ex['input'].split() else 0 for ex in examples]
axes[1, 1].scatter(avg_word_lengths, scores, color='green', alpha=0.6, s=100)
axes[1, 1].set_xlabel('Average Word Length (chars)')
axes[1, 1].set_ylabel('Readability Score')
axes[1, 1].set_title('Avg Word Length vs Readability Score')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics table
print("\n" + "="*80)
print("TEXT CHARACTERISTICS SUMMARY")
print("="*80)
print(f"{'Index':<6} {'Score':<10} {'Words':<8} {'Sentences':<12} {'Avg Word Len':<15}")
print("-"*80)
for i, ex in enumerate(examples):
    score = float(ex['output'])
    wc = count_words(ex['input'])
    sc = count_sentences(ex['input'])
    awl = np.mean([len(w) for w in ex['input'].split()]) if ex['input'].split() else 0
    print(f"{i:<6} {score:<10.3f} {wc:<8} {sc:<12} {awl:<15.2f}")

## Example Texts by Readability Level

Let's look at the actual text examples to understand what different readability scores mean in practice.

In [ ]:
# Display example texts organized by readability level
print("="*80)
print("READABILITY EXAMPLES")
print("="*80)

# Sort examples by score
sorted_examples = sorted(examples, key=lambda x: float(x['output']))

for ex in sorted_examples:
    score = float(ex['output'])
    text = ex['input']
    grade_level = ex.get('metadata_original_grade_level', 'N/A')
    
    # Categorize readability
    if score < 0.3:
        category = "EASY"
    elif score < 0.6:
        category = "MEDIUM"
    else:
        category = "HARD"
    
    print(f"\n[{category}] Score: {score:.3f} (Grade Level: {grade_level})")
    print("-"*80)
    print(text)
    print("-"*80)

print("\n" + "="*80)
print("End of demo")
print("="*80)